# PythonOS-nano — free-GPU proof run (Colab / Kaggle T4)

Turns the *prove-then-add* plan into a fundable results table on a **free T4**, with **zero funding**.

Pipeline: build a small frozen four-domain corpus → run the dense baseline → run each architecture block on the **same slice, same seed, same budget** → emit one val-loss delta table.

**What this proves:** each block is correct, stable, and worth (or not worth) its parameters.
**What it does NOT prove:** beating any external model — that needs scale + post-training. Report it as exactly that.

> Runtime → Change runtime type → **GPU (T4)** before running.

In [ ]:
# 1. GPU check + dependencies
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime type to GPU (T4)'
print('GPU:', torch.cuda.get_device_name(0))
!pip -q install tiktoken datasets tqdm numpy

In [ ]:
# 2. Get the repo.
# Option A (private repo): zip pythonos-nano, upload via the Files panel, then:
#     !unzip -q pythonos-nano.zip
# Option B (git): replace the URL with your remote.
import os
REPO = 'pythonos-nano'
if not os.path.isdir(REPO):
    # !git clone https://github.com/<you>/pythonos-nano.git
    raise SystemExit('Upload/clone pythonos-nano first (see comment above), then re-run.')
%cd $REPO
!python -c "import pythonos, sys; print('pythonos importable:', pythonos.__file__)"

In [ ]:
# 3. Build a SMALL frozen four-domain corpus (~20M tokens: minutes on a T4).
# Some source datasets are gated (need `huggingface-cli login`); any that fail
# to load are skipped and recorded in the manifest. Edit the DOMAINS table in
# data/pythonos_multidomain/prepare.py to swap datasets.
!python data/pythonos_multidomain/prepare.py --target 20e6
!echo '--- manifest ---' && cat data/pythonos_multidomain/manifest.json

## 4. Run the prove-then-add sweep

`--max_iters 1500` keeps the whole sweep to a couple of hours on a T4. The budget is identical for every block, so the comparison is valid — a short budget gives noisier deltas, not biased ones. For a **headline** number, re-run with `--max_iters 6100`.

Runs: `A_dense → B1 RoPE/NoPE → B2 SWA → B3 MLA → B4 KV-share → C1 MoE → E2 HC → E3 mHC`.

In [ ]:
!python scripts/proof_sweep.py \
    --dataset pythonos_multidomain \
    --max_iters 1500 \
    --device cuda \
    --t4

In [ ]:
# 5. Show the fundable artifact
from IPython.display import Markdown, display
with open('runs/proof_results.md', encoding='utf-8') as f:
    display(Markdown(f.read()))

### What to do with the table
- **`✅ helps`** on a param-matched row (B2, C1-active) is a real, attributable win — keep the block.
- **B3/B4** remove parameters: read the param columns first; compare B3 to **B1**, not the dense baseline.
- **E3 (mHC):** a near-zero delta is the expected near-identity collapse. Confirm at the full 6100-iter budget before concluding it can't learn — that's a *finding*, not a failure, and reporting it honestly makes the other rows more credible.
- Download `runs/proof_results.md` + `runs/proof_results.json` — those are your pre-funding evidence pack.